# CLIP for Humanists

A tool to aid the discovery process of visual semiotical analysis and geosemiotical analysis using CLIP (Contrastive Language-Image Pre-training).

This notebook is designed for non-technical users who want to analyze images using AI without writing code.

## What This Tool Does

1. **Extract GPS data** from your images to visualize geographical locations
2. **Compare images with keywords/phrases** using CLIP (an AI model that understands both images and text)
3. **Visualize trends and correlations** between images and text
4. Help identify patterns that might be useful for semiotical analysis

## How to Use This Notebook

1. Run each cell in order by clicking the play button ▶️ or pressing Shift+Enter
2. Follow the instructions in each section
3. Upload your images when prompted
4. Customize the keywords for your analysis
5. Explore the visualizations and results

Let's get started!

## Step 1: Setup

First, we need to install the necessary libraries. Run the cell below by clicking the play button ▶️ or pressing Shift+Enter.

This might take a few minutes, but you only need to do it once per session.

In [ ]:
# Install required packages
!pip install torch transformers pillow numpy pandas matplotlib folium geopy tqdm ipywidgets scikit-learn scipy

# Clone the repository
!git clone https://github.com/yourusername/CLIP_for_humanists.git
!cd CLIP_for_humanists && pip install -e .

## Step 2: Import the Code

Now we'll import the necessary code. Don't worry about understanding this part - it's just setting things up for you.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display, HTML

# Add the repository to the path
sys.path.append('/content/CLIP_for_humanists')

# Import our modules
from src.main import ClipForHumanists

# Create output directories
!mkdir -p result_images

# Initialize the system
cfh = ClipForHumanists()

print("Setup complete! You're ready to analyze images.")

## Step 3: Upload Your Images

Now it's time to upload the images you want to analyze. 

1. Run the cell below
2. Click the "Choose Files" button that appears
3. Select all the images you want to analyze
4. Wait for the upload to complete

**Note:** If your images contain GPS data (like photos taken with smartphones), this information will be used for geographical visualization.

In [ ]:
# Create a directory for uploaded images
!mkdir -p uploaded_images

# Upload images
uploaded = files.upload()

# Save uploaded files to the directory
for filename, content in uploaded.items():
    with open(os.path.join('uploaded_images', filename), 'wb') as f:
        f.write(content)

# Count the number of uploaded images
image_count = len([f for f in os.listdir('uploaded_images') if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f"\nSuccessfully uploaded {image_count} images to 'uploaded_images' folder.")

## Step 4: Define Keywords for Analysis

Now, let's define the keywords or phrases you want to compare with your images. These should be concepts relevant to your semiotical analysis.

Examples:
- For analyzing warning signs: "danger", "warning", "caution", "safety"
- For analyzing advertisements: "luxury", "affordable", "family-friendly", "modern"
- For analyzing public spaces: "welcoming", "exclusive", "formal", "casual"

You can edit the list below to include your own keywords.

In [ ]:
# Define your keywords here
keywords = [
    "scary",
    "friendly",
    "warning",
    "information",
    "official",
    "casual",
    "professional"
]

# Display the keywords
print("You've selected the following keywords for analysis:")
for keyword in keywords:
    print(f"- {keyword}")

# Optional: Add more keywords interactively
add_more = input("\nWould you like to add more keywords? (yes/no): ")
if add_more.lower() in ['yes', 'y']:
    while True:
        new_keyword = input("Enter a new keyword (or 'done' to finish): ")
        if new_keyword.lower() == 'done':
            break
        keywords.append(new_keyword)
    
    print("\nUpdated keywords for analysis:")
    for keyword in keywords:
        print(f"- {keyword}")

## Step 5: Process the Images

Now we'll process your images using CLIP to compare them with the keywords you've selected.

This might take a few minutes depending on how many images you've uploaded. The cell below will show progress as it processes each image.

In [ ]:
# Process the images
print("Processing images... This might take a few minutes.")
results = cfh.process_images('uploaded_images', keywords)
print(f"\nProcessing complete! Analyzed {len(results)} images.")

# Save the results
cfh.save_results('analysis_results.json')
print("Results saved to 'analysis_results.json'")

## Step 6: Visualize the Results

Now let's create various visualizations to help you understand the results.

### 6.1: Image and Similarity Bars

First, let's look at each image alongside a bar graph showing its similarity scores for each keyword. This gives you a clear visual representation of how each image relates to the concepts you're interested in.

In [ ]:
# Create visualizations of images with similarity bar graphs
print("Creating image similarity visualizations...")
output_paths = cfh.create_all_image_similarity_bars()
print(f"Created {len(output_paths)} visualizations in 'result_images' folder")

# Display a few examples
for i, path in enumerate(output_paths[:3]):  # Show first 3 images
    plt.figure(figsize=(12, 6))
    img = plt.imread(path)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

### 6.2: Similarity Heatmap

This heatmap shows how similar each image is to each keyword. Brighter colors indicate higher similarity.

In [ ]:
# Create a heatmap of image-text similarities
heatmap_fig = cfh.create_similarity_heatmap(title="Image-Keyword Similarity Heatmap")
plt.figure(figsize=(12, 8))
plt.imshow(heatmap_fig)
plt.show()

### 6.3: Concept Correlation Matrix

Let's look at how different concepts correlate with each other across your images. This can reveal interesting patterns, such as which concepts tend to appear together or which are opposed to each other.

In [ ]:
# Create a correlation matrix between different concepts
print("Analyzing correlations between concepts...")
corr_fig = cfh.create_similarity_correlation_matrix(title="Correlation Between Concepts")
plt.figure(figsize=(10, 8))
plt.imshow(corr_fig)
plt.show()

# Find and display the highest correlations
# Extract all unique prompts from the data
all_prompts = set()
for item in cfh.image_data:
    all_prompts.update(item["similarities"].keys())
all_prompts = sorted(list(all_prompts))

# Create a DataFrame with similarity scores for each prompt
data = []
for item in cfh.image_data:
    row = {}
    for prompt in all_prompts:
        if prompt in item["similarities"]:
            row[prompt] = item["similarities"][prompt]
        else:
            row[prompt] = float('nan')
    data.append(row)

df = pd.DataFrame(data)
corr_matrix = df.corr()

# Print the highest correlations
print("\nStrongest correlations between concepts:")
# Find the top correlations (excluding self-correlations)
high_corr = [(corr_matrix.index[i], corr_matrix.columns[j], corr_matrix.iloc[i, j])
             for i in range(len(corr_matrix.index))
             for j in range(len(corr_matrix.columns))
             if i < j]  # Only upper triangle
high_corr.sort(key=lambda x: abs(x[2]), reverse=True)
for concept1, concept2, corr in high_corr[:5]:  # Top 5 correlations
    print(f"- {concept1} and {concept2}: {corr:.3f}")

### 6.4: Geographic Map and Analysis (if GPS data is available)

If your images contain GPS data, this map will show where they were taken. We'll also analyze how the location correlates with the similarity scores.

In [ ]:
# Check if any images have GPS coordinates
has_gps = any(item.get("gps_coordinates") is not None for item in cfh.image_data)

if has_gps:
    # Create a map with images at their GPS coordinates
    print("Creating map with images...")
    map_viz = cfh.create_map(zoom_start=13)
    display(map_viz)
    
    # Perform location-based analysis
    print("\nPerforming location-based analysis...")
    
    # Normalize coordinates for analysis
    cfh.normalize_coordinates()
    
    # Cluster locations
    n_clusters = min(5, len([img for img in cfh.image_data if img.get("gps_coordinates") is not None]))
    cfh.cluster_locations(n_clusters=n_clusters)
    print(f"Clustered locations into {n_clusters} groups")
    
    # For the first prompt, create a location correlation plot as an example
    selected_prompt = keywords[0]
    print(f"\nCreating location correlation plot for '{selected_prompt}'...")
    loc_fig = cfh.create_location_correlation_plot(selected_prompt, 
                                             title=f"Geographic Distribution of '{selected_prompt}'")
    plt.figure(figsize=(10, 8))
    plt.imshow(loc_fig)
    plt.show()
    
    # Calculate correlations between location and similarity
    print("\nCalculating correlations between location and similarity scores...")
    loc_corr_df = cfh.calculate_location_similarity_correlation(keywords)
    
    # Display the correlations in a table
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 120)
    display(loc_corr_df)
    
    # Highlight any statistically significant correlations
    significant_corrs = loc_corr_df[
        (loc_corr_df['latitude_p_value'] < 0.05) | 
        (loc_corr_df['longitude_p_value'] < 0.05)]
    
    if not significant_corrs.empty:
        print("\nStatistically significant geographic correlations (p < 0.05):")
        for _, row in significant_corrs.iterrows():
            if row['latitude_p_value'] < 0.05:
                print(f"- '{row['prompt']}' correlates with latitude: {row['latitude_correlation']:.3f} (p={row['latitude_p_value']:.3f})")
            if row['longitude_p_value'] < 0.05:
                print(f"- '{row['prompt']}' correlates with longitude: {row['longitude_correlation']:.3f} (p={row['longitude_p_value']:.3f})")
    else:
        print("\nNo statistically significant geographic correlations found.")
    
    # Let user select a prompt to view its geographic distribution
    print("\nYou can view the geographic distribution for other concepts:")
    for i, keyword in enumerate(keywords):
        print(f"{i+1}. {keyword}")
    
    selection = input(f"\nEnter the number of the concept to view (1-{len(keywords)}, or 'skip'): ")
    if selection.isdigit() and 1 <= int(selection) <= len(keywords):
        selected_prompt = keywords[int(selection)-1]
        print(f"\nShowing geographic distribution for '{selected_prompt}'...")
        loc_fig = cfh.create_location_correlation_plot(selected_prompt, 
                                                 title=f"Geographic Distribution of '{selected_prompt}'")
        plt.figure(figsize=(10, 8))
        plt.imshow(loc_fig)
        plt.show()
else:
    print("No GPS data found in the uploaded images. Skipping map visualization and location analysis.")

### 6.5: Top Images for Each Keyword

For each keyword, let's see which images match it the most strongly.

In [ ]:
# For each keyword, show the top 3 matching images
for keyword in keywords:
    print(f"\n## Top images for '{keyword}':\n")
    
    # Get top images for this keyword
    top_images = cfh.get_top_images_for_prompt(keyword, n=3)
    
    # Create a grid of these images
    image_paths = [item["filepath"] for item in top_images]
    similarities = {item["filepath"]: cfh.image_data[i]["similarities"] 
                   for i, item in enumerate(cfh.image_data) 
                   if item["filepath"] in image_paths}
    
    # Display the grid
    grid = cfh.create_image_grid(image_paths, similarities, keyword, 
                               cols=3, figsize=(15, 5), 
                               title=f"Top Images for '{keyword}'")
    plt.figure(figsize=(15, 5))
    plt.imshow(grid)
    plt.show()

### 6.6: Image Grid with Similarity Scores

Let's see all your images in a grid, with their similarity scores for a specific keyword.

You can change the keyword in the cell below to see scores for different concepts.

In [ ]:
# Choose a keyword to display similarity scores for
selected_keyword = keywords[0]  # Default to the first keyword

# Let the user select a different keyword if desired
print("Available keywords:")
for i, keyword in enumerate(keywords):
    print(f"{i+1}. {keyword}")
    
selection = input(f"\nEnter the number of the keyword to display (1-{len(keywords)}, default is 1): ")
if selection.isdigit() and 1 <= int(selection) <= len(keywords):
    selected_keyword = keywords[int(selection)-1]

print(f"\nDisplaying similarity scores for '{selected_keyword}'")

# Create an image grid with similarity scores
image_paths = [item["filepath"] for item in cfh.image_data]
similarities = {item["filepath"]: item["similarities"] for item in cfh.image_data}

grid = cfh.create_image_grid(image_paths, similarities, selected_keyword, 
                           cols=3, figsize=(15, 15), 
                           title=f"All Images with Similarity Scores for '{selected_keyword}'")
plt.figure(figsize=(15, 15))
plt.imshow(grid)
plt.show()

## Step 7: Export the Results

You can download all the results as a ZIP file containing all the visualizations and data.

In [ ]:
# Convert results to a DataFrame and save as CSV
df = cfh.get_dataframe()
csv_path = 'clip_analysis_results.csv'
df.to_csv(csv_path, index=False)
print(f"Saved detailed results to {csv_path}")

# Create a ZIP file with all the results
!zip -r CLIP_analysis_results.zip *.png *.html *.csv result_images/ analysis_results.json

# Download the ZIP file
files.download('CLIP_analysis_results.zip')
print("\nAll results have been packaged into CLIP_analysis_results.zip for download.")

## Step 8: Interpreting the Results

Now that you've analyzed your images, here are some tips for interpreting the results:

1. **Similarity Scores**: These range from -1 to 1, with higher values indicating stronger association between the image and the keyword.

2. **Image-Concept Relationships**: 
   - The individual image bar charts show which concepts are most strongly associated with each image
   - Unexpected high scores might reveal hidden meanings or associations in your images
   - Look for patterns in which types of images score high for specific concepts

3. **Concept Correlations**:
   - The correlation matrix shows which concepts tend to appear together or oppose each other
   - Strong positive correlations suggest concepts that often co-occur in the same images
   - Strong negative correlations suggest concepts that tend to be mutually exclusive

4. **Geographic Patterns** (if applicable):
   - Look for clusters of similar images in specific locations
   - Geographic correlations might reveal how visual language varies by location
   - Consider if certain concepts are more prevalent in particular geographic areas

5. **Research Questions to Consider**:
   - How do similarity scores align with your own interpretations?
   - Are there geographic patterns in how certain concepts are visually represented?
   - Which images defy expectations, and what might that reveal?
   - What relationships between concepts did the correlation analysis uncover?

6. **Limitations**:
   - CLIP was trained on internet data, which may contain biases
   - The model doesn't understand cultural or historical context specific to your research
   - This tool should complement, not replace, human analysis

Remember that this tool is meant to aid discovery, not provide definitive answers. The patterns it reveals should be starting points for deeper human analysis.

## Conclusion

Congratulations! You've successfully analyzed your images using CLIP for Humanists.

You've learned how to:
1. Upload images for analysis
2. Define keywords for comparison
3. Process images with CLIP
4. Explore various visualizations of the results
5. Analyze correlations between concepts
6. Examine geographic patterns (if applicable)
7. Export the data for further analysis

We hope this tool helps you discover interesting patterns and insights for your semiotical analysis!

### Feedback and Questions

If you have feedback or questions about this tool, please visit our GitHub repository or contact us directly.

Thank you for using CLIP for Humanists!